# A tiny Karpathy-style autoresearch loop in a Daytona sandbox

Inspired by [karpathy/autoresearch](https://github.com/karpathy/autoresearch): give an AI agent a small but
real ML task and let it improve a `train.py` autonomously. The agent has a **`run_training(code)` tool** - it
calls the tool with a full `train.py`, we run it **inside a [Daytona](https://www.daytona.io) sandbox** under a
fixed time budget, **keep it only if it improved** (the "ratchet"), and feed the result back so it tries again.

## 1. Setup

```bash
pip install daytona python-dotenv langchain-fireworks
```

`.env` (repo root): `FIREWORKS_API_KEY`, `DAYTONA_API_KEY`.

In [1]:
import os, re, textwrap
import dotenv
from langchain_fireworks import ChatFireworks
from daytona import Daytona, DaytonaConfig, CreateSandboxFromSnapshotParams

dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True), override=True)
assert os.getenv("FIREWORKS_API_KEY"), "FIREWORKS_API_KEY missing from .env"
assert os.getenv("DAYTONA_API_KEY"), "DAYTONA_API_KEY missing from .env"

MODEL = "accounts/fireworks/models/glm-5p2"   # serverless: nothing to deploy
N_ITERS = 5                                     # how many propose/run/keep cycles
RUN_BUDGET_S = 600                              # fixed wall-clock per experiment (the 'ratchet' constraint)
WORKDIR = "/home/daytona"

chat = ChatFireworks(
    model=MODEL,
    fireworks_api_key=os.environ["FIREWORKS_API_KEY"],
    fireworks_api_base="https://api.fireworks.ai/inference",  # client appends /v1/chat/completions
    temperature=0.6,
    max_tokens=None,
)
print("model:", MODEL)

/Users/sinan/cookbook-casestudies/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model: accounts/fireworks/models/glm-5p2


## 2. Spin up the sandbox + seed the task

We boot one stock Daytona sandbox and download a **real** dataset into it:
[`sealuzh/app_reviews`](https://huggingface.co/datasets/sealuzh/app_reviews) — app-store reviews with a 1-5
`star` rating. The task: **predict `star` from the `review` text** (5-class text classification). We subsample
a fixed slice into `train.parquet` / `val.parquet` (immutable across iterations so results are comparable).
We do **not** write any starter code — the agent writes `train.py` from scratch (we only describe the contract).

In [2]:
daytona = Daytona(DaytonaConfig(api_key=os.environ["DAYTONA_API_KEY"]))
# auto_stop/pause/archive = 0 -> sandbox stays up across cells (no mid-run stops).
sandbox = daytona.create(CreateSandboxFromSnapshotParams(
    snapshot="daytona-medium", auto_stop_interval=0, auto_pause_interval=0, auto_archive_interval=0,
), timeout=120)
print("sandbox up")

def sh(cmd, timeout=RUN_BUDGET_S):
    """Run a command; auto-restart a stopped sandbox once, and never raise (so a failed
    experiment just becomes a revert in the loop)."""
    for attempt in range(2):
        try:
            r = sandbox.process.exec(cmd, cwd=WORKDIR, timeout=timeout)
            return int(getattr(r, "exit_code", 0) or 0), str(getattr(r, "result", "") or "")
        except Exception as e:
            if attempt == 0:
                try: sandbox.start()
                except Exception: pass
                continue
            return 1, f"[sandbox exec error] {str(e)[:300]}"

def put(path, content):
    sandbox.fs.upload_file(content.encode(), path)

# datasets + pyarrow to pull the HF dataset and write parquet inside the sandbox.
print(sh("pip install -q datasets pyarrow", timeout=300)[1][-300:])

# Immutable data (like autoresearch's prepare.py) -- built once, never edited by the agent.
put(f"{WORKDIR}/make_data.py", textwrap.dedent('''
    import pandas as pd
    from datasets import load_dataset
    from sklearn.model_selection import train_test_split
    df = load_dataset("sealuzh/app_reviews", split="train").to_pandas()[["review", "star"]].dropna()
    n = 5_000
    if len(df) > n:  # small stratified sample keeps each experiment fast
        _, df = train_test_split(df, test_size=n, random_state=0, stratify=df["star"])
    tr, va = train_test_split(df, test_size=0.2, random_state=0, stratify=df["star"])
    tr.to_parquet("/home/daytona/train.parquet")
    va.to_parquet("/home/daytona/val.parquet")
    print("data ready", tr.shape, va.shape, "| star classes:", sorted(df["star"].unique()))
'''))
print(sh("python make_data.py", timeout=300)[1])

sandbox up
s-cli is installed in '/home/daytona/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

Generating train split: 100%|██████████| 288065/288065 [00:00<00:00, 523679.22 examples/s]
data ready (4000, 2) (1000, 2) | star classes: [np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5)]



## 3. Give the agent a `run_training` tool

Instead of parsing code blocks out of the model's text, we give it a real **tool**: `run_training(code)`. The
model calls it with a full `train.py` as a structured argument; we write that file into the sandbox, run it
under the fixed budget, keep it if `VAL_ACC` improved, and hand the result back as a tool message. The agent
then decides what to do next — write another `train.py` or stop.

In [3]:
from langchain.agents import create_agent
from langchain_core.tools import tool

# Shared state the tool updates (the 'ratchet': keep the best train.py seen so far).
state = {"best_acc": None, "best_code": None, "history": [], "agent_runs": 0}

def _run_training(code):
    """Write train.py, run it, update best, return feedback text."""
    put(f"{WORKDIR}/train.py", code)
    _, out = sh("python train.py")
    m = re.search(r"VAL_ACC=([0-9.]+)", out)
    acc = float(m.group(1)) if m else None
    if acc is None:
        state["history"].append(None)
        return f"Run FAILED (no VAL_ACC printed). Output tail:\n{out.strip()[-1200:]}\nFix it and try again."
    kept = state["best_acc"] is None or acc > state["best_acc"]
    if kept:
        state["best_acc"], state["best_code"] = acc, code
    state["history"].append(acc)
    return (f"VAL_ACC={acc:.4f}. Best so far={state['best_acc']:.4f}. "
            f"{'Kept (new best).' if kept else 'Did NOT beat best; best is unchanged.'}")

@tool
def run_training(code: str) -> str:
    """Write the COMPLETE train.py source to the sandbox, run it, and return the validation result (VAL_ACC).
    Use this to try an improved train.py."""
    state["agent_runs"] += 1
    if state["agent_runs"] > N_ITERS:
        return "Experiment budget reached. Do not call run_training again; give a short final summary."
    fb = _run_training(code)
    print(f"  experiment {state['agent_runs']}: {fb.splitlines()[0]}")
    return fb

SYSTEM = (
    "You are an ML research agent. Goal: MAXIMIZE validation accuracy on a TEXT classification task — predict "
    "the 1-5 'star' rating from the 'review' text of app-store reviews. Use the run_training tool: pass a "
    "COMPLETE train.py that reads /home/daytona/train.parquet and /home/daytona/val.parquet (columns: "
    "review [str], star [int]), trains ONLY on train, evaluates accuracy on val, and prints exactly one line "
    f"'VAL_ACC=<float>'. Each run must finish within ~{RUN_BUDGET_S}s on CPU (pandas / scikit-learn; classic "
    "NLP like TF-IDF, n-grams, linear/tree models). Call run_training repeatedly to beat the best score, using "
    "the returned feedback to guide the next attempt. Stop when improvements plateau."
)

# LangChain handles the whole tool loop for us.
agent = create_agent(model=chat, tools=[run_training], system_prompt=SYSTEM)

## 4. Run the agent

No starter code: the agent writes `train.py` itself on its first `run_training` call, then iterates — keeping
the best, using the returned score to guide the next attempt. Watch the best score climb.

In [4]:
from langchain_core.messages import HumanMessage

# No starter code -- the agent writes train.py from scratch on its first run_training call.
intro = (
    "Write train.py from scratch and call run_training with it. The data is already at "
    "/home/daytona/train.parquet and /home/daytona/val.parquet (columns: review [str], star [int 1..5]). "
    "Start with a simple, fast baseline (e.g. TF-IDF + logistic regression) to get a first score, then iterate "
    f"to beat it. You have {N_ITERS} experiments."
)
agent.invoke({"messages": [HumanMessage(intro)]}, config={"recursion_limit": N_ITERS * 2 + 6})
print(f"\nbest VAL_ACC = {state['best_acc']}")

  experiment 1: Run FAILED (no VAL_ACC printed). Output tail:
  experiment 2: VAL_ACC=0.6550. Best so far=0.6550. Kept (new best).
  experiment 3: VAL_ACC=0.6590. Best so far=0.6590. Kept (new best).
  experiment 4: VAL_ACC=0.6620. Best so far=0.6620. Kept (new best).
  experiment 5: VAL_ACC=0.6260. Best so far=0.6620. Did NOT beat best; best is unchanged.

best VAL_ACC = 0.662


## 5. Results

The improvement log and the winning `train.py` the agent discovered.

In [5]:
print("=== score history (each agent experiment) ===")
for i, acc in enumerate(state["history"], 1):
    print(f"  exp {i:<3} {acc}")
print(f"\nbest VAL_ACC = {state['best_acc']}")
print("\n=== best train.py ===\n")
print(state["best_code"] or "(no successful run)")

=== score history (each agent experiment) ===
  exp 1   None
  exp 2   0.655
  exp 3   0.659
  exp 4   0.662
  exp 5   0.626

best VAL_ACC = 0.662

=== best train.py ===


import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import scipy.sparse as sp

train = pd.read_parquet('/home/daytona/train.parquet')
val = pd.read_parquet('/home/daytona/val.parquet')

X_train = train['review'].astype(str).values
y_train = train['star'].values
X_val = val['review'].astype(str).values
y_val = val['star'].values

# Richer features: word (1-3) + char (3-5), higher max_features
word_vec = TfidfVectorizer(ngram_range=(1,3), max_features=150000, sublinear_tf=True, min_df=2)
char_vec = TfidfVectorizer(ngram_range=(3,5), analyzer='char_wb', max_features=150000, sublinear_tf=True, min_df=2)

Xtr_w = word_vec.fit_transform(X_train)
Xtr_c = char_vec.fit_transform(X_train)
Xva_w = word_vec.

## 6. Clean up

Delete the sandbox.

In [6]:
sandbox.delete()
print("sandbox deleted")

sandbox deleted
